In [1]:
import gymnasium as gym

import torch

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


# In this Notebook
* Load trained baseline model
* get_parameters()
* get value function values for a trajectory

## PPO 

PPO is an actor-critic method, where it learns a policy $\pi(a|s)$ and evlauates it by a state-value function $V(s)$.

Thorugh `base_lines` policy module, we can obtain a models value function through `predict_values(s)`

Through the relation 

$$Q(s,a) = E_{s'} [R + \gamma V(s')]$$

$$Q(s,a) = \sum_{s'} P(s'|s, a) [R(s,a, s') + \gamma V(s')]$$

Or, in the case of a deterministic policy, where it always selects one action: 

$$
V^{\pi}(s) = Q^\pi (s, \pi(s))
$$


Neither Q values nor transition probabilities are available; instead we make use of the one-step TD target

$$r + \gamma V(s')$$

In [2]:
model = PPO.load(r"..\code\baseline_code\baseline_models\ppo_cartpole.zip")

c:\Users\sofie\anaconda3\envs\thesis\lib\site-packages\stable_baselines3\common\save_util.py:166: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'c:\\Users\\sofie\\anaconda3\\envs\\thesis\\lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(
c:\Users\sofie\anaconda3\envs\thesis\lib\site-packages\stable_baselines3\common\save_util.py:166: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'c:\\Users\\sofie\\anaconda3\\envs\\thesis\\lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(


In [3]:
vec_env = make_vec_env("CartPole-v1", n_envs=1)

In [4]:
obs = vec_env.reset()
print(f"current state values are: {obs}")

action_one = torch.tensor([1]).to("cuda")
action_zero = torch.tensor([0]).to("cuda")

opposite = {1 : action_zero, 0 : action_one}

while True:
    action, _states = model.predict(obs)
    print(f"chosen action is: {action}")
    t_obs = torch.from_numpy(obs).to("cuda")
    t_action = torch.from_numpy(action).to("cuda")

    state_value = model.policy.predict_values(t_obs)
    action_value = model.policy.evaluate_actions(t_obs, t_action)
    print(f"value of state is {state_value.item()}")
    print(f"value of chosen action is {action_value[0].item()}")

    print(f"opposite {opposite[t_action.item()].item()}")

    opp_action_value = model.policy.evaluate_actions(t_obs, opposite[t_action.item()])
    print(f"value of opposite action is {opp_action_value[0].item()}")

    obs, rewards, dones, info = vec_env.step(action)
    if dones: 
        break


current state values are: [[-0.02617051 -0.0068043   0.01824846  0.00328613]]
chosen action is: [1]
value of state is 55.548152923583984
value of chosen action is 55.548152923583984
opposite 0
value of opposite action is 55.548152923583984
chosen action is: [0]
value of state is 55.53369140625
value of chosen action is 55.53369140625
opposite 1
value of opposite action is 55.53369140625
chosen action is: [1]
value of state is 55.552001953125
value of chosen action is 55.552001953125
opposite 0
value of opposite action is 55.552001953125
chosen action is: [0]
value of state is 55.53862762451172
value of chosen action is 55.53862762451172
opposite 1
value of opposite action is 55.53862762451172
chosen action is: [0]
value of state is 55.55552673339844
value of chosen action is 55.55552673339844
opposite 1
value of opposite action is 55.55552673339844
chosen action is: [1]
value of state is 55.421539306640625
value of chosen action is 55.421539306640625
opposite 0
value of opposite action

In [5]:
with torch.no_grad():
    t_obs = torch.as_tensor(obs).to(model.device)

    # State value V(s)
    state_value = model.policy.predict_values(t_obs)

    # Action distribution π(a|s)
    distribution = model.policy.get_distribution(t_obs)
    probs = distribution.distribution.probs

print("V(s):", state_value.item())
print("Action probabilities:", probs.cpu().numpy())


V(s): 55.561912536621094
Action probabilities: [[0.55168605 0.44831392]]


In [8]:
import numpy as np
import gymnasium as gym
from stable_baselines3.common.evaluation import evaluate_policy

from imitation.algorithms import bc
from imitation.data import rollout
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.policies.serialize import load_policy
from imitation.util.util import make_vec_env

In [ ]:
rng = np.random.default_rng(0)

env = make_vec_env(
    "CartPole-v1",
    rng=rng,
    n_envs=1,
    post_wrappers=[lambda env, _: RolloutInfoWrapper(env)],  # <-- important
)

expert = PPO.load(r"..\code\baseline_code\baseline_models\ppo_cartpole.zip")

rollouts = rollout.rollout(
    expert.policy,
    env,
    rollout.make_sample_until(min_timesteps=None, min_episodes=50),
    rng=rng,
)

transitions = rollout.flatten_trajectories(rollouts)

bc_trainer = bc.BC(
    observation_space=env.observation_space,
    action_space=env.action_space,
    demonstrations=transitions,
    rng=rng,
)
bc_trainer.train(n_epochs=1)

reward, _ = evaluate_policy(bc_trainer.policy, env, 10)
print("Reward:", reward)


0batch [00:00, ?batch/s]

---------------------------------
| batch_size        | 32        |
| bc/               |           |
|    batch          | 0         |
|    ent_loss       | -0.000693 |
|    entropy        | 0.693     |
|    epoch          | 0         |
|    l2_loss        | 0         |
|    l2_norm        | 72.5      |
|    loss           | 0.693     |
|    neglogp        | 0.693     |
|    prob_true_act  | 0.5       |
|    samples_so_far | 32        |
---------------------------------


459batch [00:04, 107.64batch/s]


Reward: 500.0


In [16]:
reward, _ = evaluate_policy(bc_trainer.policy, env, 100)
print("Reward:", reward)

Reward: 500.0
